# Cross-Industry Accelerator — Create Ontology
### Auto-Create Fabric IQ Ontology from Semantic Model

This notebook creates a **Fabric IQ Ontology** item by reading the Power BI Semantic Model
created in notebook 04 and transforming its structure into an ontology.

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Fetches the Semantic Model BIM definition via Fabric REST API
3. Transforms BIM tables → Entity Types, BIM columns → Properties, BIM relationships → Relationship Types
4. Binds entities to Warehouse tables and relationships to fact tables
5. Creates the ontology item in Fabric with all data bindings
6. Displays entity types, relationship types, and binding summary

> **Prerequisites:**
> 1. Run `03_Load_Warehouse.ipynb` so warehouse tables exist for data bindings
> 2. Run `04_Create_Semantic_Model.ipynb` to create the semantic model
> 3. (Optional) Configure Eventhouse for TimeSeries bindings

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEM IDs
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
if lh_matches.empty:
    print(f"⚠ Lakehouse '{LAKEHOUSE_NAME}' not found in workspace. Available:")
    print(items_df[items_df["Type"] == "Lakehouse"][["Display Name", "Id"]].to_string(index=False))
    binding_lakehouse_item_id = ""
else:
    binding_lakehouse_item_id = str(lh_matches.iloc[0].Id)
    print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {binding_lakehouse_item_id}")

# Resolve Eventhouse item ID (optional)
binding_eventhouse_item_id = ""
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if eh_matches.empty:
        print(f"⚠ Eventhouse '{EVENTHOUSE_NAME}' not found. Ontology will be created without Eventhouse bindings.")
    else:
        binding_eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {binding_eventhouse_item_id}")
else:
    print("ℹ No Eventhouse configured — Eventhouse bindings will be skipped.")

print(f"\n✅ Workspace ID: {workspace_id}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# GENERATE ONTOLOGY FROM SEMANTIC MODEL 
# ════════════════════════════════════════════════════════════════════════
# Fetches the BIM model definition from the semantic model and builds
# the ontology structure directly from it — ensuring consistency.

import base64, json, random, uuid, time
import requests as req_lib
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
api_headers = {"Authorization": f"Bearer {access_token}", "Content-Type": "application/json"}
FABRIC_API = "https://api.fabric.microsoft.com/v1"

# Get Warehouse ID
items_df = fabric.list_items()
wh_match = items_df[(items_df["Type"].isin(["DataWarehouse", "Warehouse"])) & (items_df["Display Name"] == WAREHOUSE_NAME)]
WAREHOUSE_ID = str(wh_match.iloc[0].Id)
WAREHOUSE_SCHEMA = "dbo"

# Get Semantic Model ID
sm_match = items_df[(items_df["Type"] == "SemanticModel") & (items_df["Display Name"] == SEMANTIC_MODEL_NAME)]
if sm_match.empty:
    raise RuntimeError(f"Semantic Model '{SEMANTIC_MODEL_NAME}' not found. Run 04_Create_Semantic_Model.ipynb first.")
SEMANTIC_MODEL_ID = str(sm_match.iloc[0].Id)

# Get KQL Database ID (for TimeSeries bindings)
kql_db_id = None
try:
    kql_resp = req_lib.get(f"{FABRIC_API}/workspaces/{workspace_id}/items?type=KQLDatabase", headers=api_headers)
    if kql_resp.ok:
        for item in kql_resp.json().get("value", []):
            if EVENTHOUSE_DATABASE.lower() in item["displayName"].lower():
                kql_db_id = item["id"]
                break
except Exception:
    pass

print(f"Semantic Model: {SEMANTIC_MODEL_NAME} -> {SEMANTIC_MODEL_ID}")
print(f"Warehouse:      {WAREHOUSE_NAME} -> {WAREHOUSE_ID}")
print(f"KQL DB:         {kql_db_id or '(not found)'}\n")

# ── Fetch Semantic Model Definition (with fallback to CSV parsing) ──
print("Fetching semantic model definition...")
bim_model = None
bim_tables = []
bim_relationships = []

try:
    sm_def_resp = req_lib.post(
        f"{FABRIC_API}/workspaces/{workspace_id}/semanticModels/{SEMANTIC_MODEL_ID}/getDefinition",
        headers=api_headers,
        timeout=60
    )

    if sm_def_resp.status_code == 202:
        # Async operation - poll for 2 minutes max
        op_url = sm_def_resp.headers.get("Location", "")
        if op_url:
            print(f"  Definition export in progress...")
            for attempt in range(24):  # 2 minutes max
                time.sleep(5)
                poll_resp = req_lib.get(op_url, headers=api_headers, timeout=30)
                if poll_resp.ok:
                    result = poll_resp.json()
                    status = result.get("status", "")
                    if attempt % 6 == 0:
                        print(f"    Status: {status} (attempt {attempt+1}/24)")
                    
                    if status in ("Succeeded", "Completed"):
                        if "result" in result:
                            sm_def_resp = type('obj', (object,), {'status_code': 200, 'json': lambda: result["result"]})()
                        else:
                            time.sleep(2)
                            sm_def_resp = req_lib.post(
                                f"{FABRIC_API}/workspaces/{workspace_id}/semanticModels/{SEMANTIC_MODEL_ID}/getDefinition",
                                headers=api_headers, timeout=60
                            )
                        if sm_def_resp.status_code == 200:
                            break
                    elif status in ("Failed", "Cancelled"):
                        raise RuntimeError(f"Definition export failed: {result.get('error', {}).get('message', str(result))}")
            else:
                raise RuntimeError("Timeout after 2 minutes - falling back to CSV parsing")

    if sm_def_resp.status_code == 200:
        # Parse BIM model from definition
        for part in sm_def_resp.json().get("definition", {}).get("parts", []):
            if part["path"] == "model.bim":
                payload = part.get("payload", "")
                bim_json = base64.b64decode(payload).decode("utf-8")
                bim_model = json.loads(bim_json)
                bim_tables = bim_model.get("model", {}).get("tables", [])
                bim_relationships = bim_model.get("model", {}).get("relationships", [])
                print(f"  ✓ BIM model fetched: {len(bim_tables)} tables, {len(bim_relationships)} relationships")
                break
    
    if not bim_model:
        raise RuntimeError("No model.bim found in semantic model definition")

except Exception as e:
    print(f"  ⚠ Could not fetch semantic model: {e}")
    print(f"  → Falling back to CSV parsing...")
    bim_model = None

# ── Deterministic ID generation ──
_rng = random.Random(42)
def _new_id(): return str(_rng.randint(10**12, 10**18))
def _b64(obj): return base64.b64encode(json.dumps(obj).encode()).decode()

# ── Type mappings ──
_TYPE_MAP = {"string": "String", "int64": "BigInt", "double": "Double", "dateTime": "DateTime", "boolean": "String"}
def _onto_type(bim_type): return _TYPE_MAP.get(bim_type, "String")

def _to_entity_name(table_name):
    name = table_name.replace("dim_", "").replace("fact_", "")
    parts = name.split("_")
    pascal = "".join(p.capitalize() for p in parts)
    if pascal.endswith("ies"): pascal = pascal[:-3] + "y"
    elif pascal.endswith("s") and not pascal.endswith("ss"): pascal = pascal[:-1]
    return pascal

# ── Parse BIM tables into dim/fact structures (from BIM or CSV fallback) ──
dim_tables = {}
fact_tables = {}

if bim_model:
    # Parse from BIM model
    for bim_table in bim_tables:
        table_name = bim_table["name"]
        cols = [{"name": c["name"], "dataType": c.get("dataType", "string")} for c in bim_table.get("columns", [])]
        
        if table_name.startswith("dim_"):
            key_col = next((c["name"] for c in cols if c["name"].endswith("_id")), None)
            display_col = next((c["name"] for c in cols if c["name"] in ("name", "first_name")), key_col)
            dim_tables[table_name] = {"columns": cols, "key_col": key_col, "display_col": display_col}
        else:
            ts_col = next((c["name"] for c in cols if "timestamp" in c["name"].lower() or "date" in c["name"].lower()), None)
            # Extract FK columns from BIM relationships
            fk_cols = {}
            for rel in bim_relationships:
                if rel.get("fromTable") == table_name:
                    fk_col = rel.get("fromColumn")
                    target_table = rel.get("toTable")
                    if fk_col and target_table:
                        fk_cols[fk_col] = target_table
            fact_tables[table_name] = {"columns": cols, "timestamp_col": ts_col, "fk_cols": fk_cols}
    
    print(f"✓ Parsed from BIM: {len(dim_tables)} dim tables, {len(fact_tables)} fact/stream tables")
else:
    # Fallback: Parse from CSVs
    print("  Parsing table schemas from CSVs...")
    for table_name in DIM_TABLES:
        csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
        try:
            df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
            cols = []
            for f in df.schema.fields:
                dt = type(f.dataType).__name__
                bim_dt = {"StringType": "string", "IntegerType": "int64", "LongType": "int64", "FloatType": "double", "DoubleType": "double", "BooleanType": "string", "DateType": "string", "TimestampType": "string"}.get(dt, "string")
                cols.append({"name": f.name, "dataType": bim_dt})
            key_col = next((c["name"] for c in cols if c["name"].endswith("_id")), None)
            display_col = next((c["name"] for c in cols if c["name"] in ("name", "first_name")), key_col)
            dim_tables[table_name] = {"columns": cols, "key_col": key_col, "display_col": display_col}
        except Exception as e:
            print(f"    ✗ {table_name}: {e}")
    
    for table_name in (FACT_TABLES_BATCH + FACT_TABLES_EVENT + STREAM_TABLES):
        csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
        try:
            df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)
            cols = []
            for f in df.schema.fields:
                dt = type(f.dataType).__name__
                bim_dt = {"StringType": "string", "IntegerType": "int64", "LongType": "int64", "FloatType": "double", "DoubleType": "double", "BooleanType": "string", "DateType": "string", "TimestampType": "string"}.get(dt, "string")
                cols.append({"name": f.name, "dataType": bim_dt})
            ts_col = next((c["name"] for c in cols if "timestamp" in c["name"].lower() or "date" in c["name"].lower()), None)
            # Auto-detect FK columns
            fk_cols = {}
            dim_keys = {dt: di["key_col"] for dt, di in dim_tables.items() if di["key_col"]}
            for c in cols:
                if c["name"].endswith("_id"):
                    for dt, dk in dim_keys.items():
                        if c["name"] == dk:
                            fk_cols[c["name"]] = dt
                            break
            fact_tables[table_name] = {"columns": cols, "timestamp_col": ts_col, "fk_cols": fk_cols}
        except Exception as e:
            print(f"    ✗ {table_name}: {e}")
    
    print(f"✓ Parsed from CSVs: {len(dim_tables)} dim tables, {len(fact_tables)} fact/stream tables")

# Discover KQL tables from Eventhouse
kql_tables = {}
if EVENTHOUSE_CLUSTER_URI:
    try:
        kusto_token = notebookutils.credentials.getToken(EVENTHOUSE_CLUSTER_URI)
        kh = {"Authorization": f"Bearer {kusto_token}", "Content-Type": "application/json"}
        r = req_lib.post(f"{EVENTHOUSE_CLUSTER_URI}/v1/rest/mgmt", headers=kh,
            json={"db": EVENTHOUSE_DATABASE, "csl": ".show tables | project TableName"})
        if r.ok:
            for row in r.json().get("Tables", [{}])[0].get("Rows", []):
                tn = row[0]
                r2 = req_lib.post(f"{EVENTHOUSE_CLUSTER_URI}/v1/rest/query", headers=kh,
                    json={"db": EVENTHOUSE_DATABASE, "csl": f"{tn} | getschema"})
                if r2.ok:
                    schema_rows = r2.json().get("Tables", [{}])[0].get("Rows", [])
                    cols = [{"name": r[0], "cslType": r[2]} for r in schema_rows]
                    ts = next((c["name"] for c in cols if c["cslType"] == "System.DateTime"), None)
                    kql_tables[tn] = {"columns": cols, "timestamp_col": ts}
        print(f"KQL Tables: {len(kql_tables)} found")
    except Exception as e:
        print(f"KQL discovery: {e}")

# ── Build ontology definition parts ──
parts = []
entity_ids = {}
entity_key_prop_ids = {}
entity_prop_lookup = {}

# .platform
parts.append({"path": ".platform", "payload": _b64({"metadata": {"type": "Ontology", "displayName": ONTOLOGY_NAME}}), "payloadType": "InlineBase64"})
parts.append({"path": "definition.json", "payload": _b64({}), "payloadType": "InlineBase64"})

# Entity Types from dim tables
for tname, tinfo in dim_tables.items():
    ent_name = _to_entity_name(tname)
    ent_id = _new_id()
    entity_ids[ent_name] = ent_id

    properties = []
    key_prop_id = display_prop_id = None
    for col in tinfo["columns"]:
        pid = _new_id()
        entity_prop_lookup[(ent_name, col["name"])] = pid
        properties.append({"id": pid, "name": col["name"], "redefines": None, "baseTypeNamespaceType": None, "valueType": _onto_type(col["dataType"])})
        if col["name"] == tinfo["key_col"]: key_prop_id = pid; entity_key_prop_ids[ent_name] = pid
        if col["name"] == tinfo["display_col"]: display_prop_id = pid

    entity_def = {
        "id": ent_id, "namespace": "usertypes", "baseEntityTypeId": None,
        "name": ent_name, "entityIdParts": [key_prop_id] if key_prop_id else [],
        "displayNamePropertyId": display_prop_id or key_prop_id,
        "namespaceType": "Custom", "visibility": "Visible",
        "properties": properties, "timeseriesProperties": [],
    }
    parts.append({"path": f"EntityTypes/{ent_id}/definition.json", "payload": _b64(entity_def), "payloadType": "InlineBase64"})

    # NonTimeSeries binding (Warehouse)
    bid = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"ontology.{ent_name}.static"))
    binding = {
        "id": bid,
        "dataBindingConfiguration": {
            "dataBindingType": "NonTimeSeries",
            "propertyBindings": [{"sourceColumnName": c["name"], "targetPropertyId": entity_prop_lookup[(ent_name, c["name"])]} for c in tinfo["columns"]],
            "sourceTableProperties": {"sourceType": "LakehouseTable", "workspaceId": workspace_id, "itemId": WAREHOUSE_ID, "sourceTableName": tname, "sourceSchema": WAREHOUSE_SCHEMA},
        },
    }
    parts.append({"path": f"EntityTypes/{ent_id}/DataBindings/{bid}.json", "payload": _b64(binding), "payloadType": "InlineBase64"})

# Relationships from BIM model (use actual BIM relationships instead of inferring)
from itertools import combinations
rel_pairs = set()

# Extract unique entity pairs from BIM relationships
for rel in bim_relationships:
    from_tbl = rel.get("toTable")  # Target is dimension
    to_tbl = rel.get("fromTable")  # Source is fact
    if from_tbl and to_tbl and from_tbl.startswith("dim_") and to_tbl:
        # Only create relationships between dimensions
        pass

# Build relationships between dimensions that are linked through facts
for ft_name, ft_info in fact_tables.items():
    linked = sorted(set(ft_info["fk_cols"].values()))
    for a, b in combinations(linked, 2):
        rel_pairs.add((a, b))

rel_count = 0
for src_tbl, tgt_tbl in sorted(rel_pairs):
    src_e, tgt_e = _to_entity_name(src_tbl), _to_entity_name(tgt_tbl)
    if src_e not in entity_ids or tgt_e not in entity_ids: continue
    rel_id = _new_id()
    rel_name = f"{src_e}To{tgt_e}"[:26]
    parts.append({"path": f"RelationshipTypes/{rel_id}/definition.json", "payload": _b64({
        "namespace": "usertypes", "id": rel_id, "name": rel_name, "namespaceType": "Custom",
        "source": {"entityTypeId": entity_ids[src_e]}, "target": {"entityTypeId": entity_ids[tgt_e]},
    }), "payloadType": "InlineBase64"})

    # Find binding table (fact table that connects these two dims)
    for ft_name, ft_info in fact_tables.items():
        src_col = next((c for c, d in ft_info["fk_cols"].items() if d == src_tbl), None)
        tgt_col = next((c for c, d in ft_info["fk_cols"].items() if d == tgt_tbl), None)
        if src_col and tgt_col:
            rb_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"ontology.rel.{rel_name}"))
            parts.append({"path": f"RelationshipTypes/{rel_id}/DataBindings/{rb_id}.json", "payload": _b64({
                "id": rb_id,
                "dataBindingTable": {"workspaceId": workspace_id, "itemId": WAREHOUSE_ID, "sourceTableName": ft_name, "sourceSchema": WAREHOUSE_SCHEMA, "sourceType": "LakehouseTable"},
                "sourceEntityKeyColumnName": src_col, "targetEntityKeyColumnName": tgt_col,
            }), "payloadType": "InlineBase64"})
            break
    rel_count += 1

et_count = sum(1 for p in parts if p["path"].startswith("EntityTypes/") and p["path"].endswith("/definition.json"))
db_count = sum(1 for p in parts if "/DataBindings/" in p["path"])
source_label = "from BIM model" if bim_model else "from CSVs"
print(f"\n✅ Ontology definition built {source_label}:")
print(f"  Entity Types:   {et_count}")
print(f"  Relationships:  {rel_count}")
print(f"  Data Bindings:  {db_count}")
print(f"  Total parts:    {len(parts)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE ONTOLOGY IN FABRIC (two-phase: structure then bindings)
# ════════════════════════════════════════════════════════════════════════

# Split into structure (no bindings) and binding parts
structure_parts = [p for p in parts if "/DataBindings/" not in p["path"]]
binding_parts = [p for p in parts if "/DataBindings/" in p["path"]]

# Check for existing ontology
existing_id = None
try:
    r = req_lib.get(f"{FABRIC_API}/workspaces/{workspace_id}/ontologies", headers=api_headers, timeout=30)
    if r.ok:
        for item in r.json().get("value", []):
            if item["displayName"] == ONTOLOGY_NAME:
                existing_id = item["id"]
                break
except Exception:
    pass

def _poll_op(op_id):
    for _ in range(30):
        time.sleep(5)
        r = req_lib.get(f"{FABRIC_API}/operations/{op_id}", headers=api_headers, timeout=30)
        if r.ok:
            s = r.json().get("status", "")
            print(f"    Status: {s}")
            if s in ("Succeeded", "Completed"): return True
            if s in ("Failed", "Cancelled"):
                print(f"    {json.dumps(r.json(), indent=2)}")
                return False
    return False

if existing_id:
    print(f"Existing ontology found: {existing_id}. Deleting...")
    dr = req_lib.delete(f"{FABRIC_API}/workspaces/{workspace_id}/ontologies/{existing_id}", headers=api_headers, timeout=60)
    if dr.status_code == 202:
        op_id = dr.headers.get("x-ms-operation-id", "")
        _poll_op(op_id)
    time.sleep(20)

# Phase 1: Create with structure only
print(f"\nPhase 1: Creating ontology structure ({ONTOLOGY_NAME})...")
create_body = {
    "displayName": ONTOLOGY_NAME,
    "description": f"Auto-generated ontology for {INDUSTRY} industry ({et_count} entities, {rel_count} rels, {db_count} bindings)",
    "definition": {"parts": structure_parts},
}

ok = False
for attempt in range(5):
    r = req_lib.post(f"{FABRIC_API}/workspaces/{workspace_id}/ontologies", headers=api_headers, json=create_body, timeout=60)
    print(f"  HTTP {r.status_code}")
    if r.status_code == 201:
        print(f"  Created: {r.json().get('id')}")
        ok = True
        break
    elif r.status_code == 202:
        op_id = r.headers.get("x-ms-operation-id", "")
        ok = _poll_op(op_id)
        break
    elif r.status_code == 400 and "NotAvailableYet" in r.text:
        print(f"  Name locked — waiting 30s (attempt {attempt+1}/5)...")
        time.sleep(30)
    elif r.status_code == 403 and "FeatureNotAvailable" in r.text:
        print("  Fabric IQ Ontology feature not enabled in this tenant.")
        print("  Enable in: Admin Portal -> Tenant settings -> Fabric IQ")
        break
    else:
        print(f"  Error: {r.text[:300]}")
        break

# Phase 2: Add data bindings
if ok and binding_parts:
    time.sleep(5)
    # Find the created ontology ID
    new_id = None
    r = req_lib.get(f"{FABRIC_API}/workspaces/{workspace_id}/ontologies", headers=api_headers, timeout=30)
    if r.ok:
        for item in r.json().get("value", []):
            if item["displayName"] == ONTOLOGY_NAME:
                new_id = item["id"]
                break
    if new_id:
        print(f"\nPhase 2: Adding data bindings to {new_id}...")
        ur = req_lib.post(f"{FABRIC_API}/workspaces/{workspace_id}/ontologies/{new_id}/updateDefinition",
            headers=api_headers, json={"definition": {"parts": parts}}, timeout=60)
        print(f"  HTTP {ur.status_code}")
        if ur.status_code == 202:
            op_id = ur.headers.get("x-ms-operation-id", "")
            _poll_op(op_id)
        elif ur.status_code == 200:
            print("  Bindings added.")
    else:
        print("  Could not find ontology ID for binding update.")

print(f"\nOntology creation complete: {ONTOLOGY_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# ONTOLOGY SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'='*60}")
print(f"ONTOLOGY SUMMARY - {INDUSTRY.upper()}")
print(f"{'='*60}")

print(f"\nEntity Types ({len(dim_tables)}):")
for tname in sorted(dim_tables):
    ent = _to_entity_name(tname)
    ncols = len(dim_tables[tname]["columns"])
    print(f"   {ent} -> {WAREHOUSE_SCHEMA}.{tname} ({ncols} properties)")

print(f"\nRelationship Types ({rel_count}):")
for src_tbl, tgt_tbl in sorted(rel_pairs):
    src_e, tgt_e = _to_entity_name(src_tbl), _to_entity_name(tgt_tbl)
    if src_e in entity_ids and tgt_e in entity_ids:
        print(f"   {src_e}To{tgt_e}: {src_e} -> {tgt_e}")

if kql_tables:
    print(f"\nTimeSeries Sources ({len(kql_tables)} KQL tables):")
    for ktn in sorted(kql_tables):
        ncols = len(kql_tables[ktn]["columns"])
        ts = kql_tables[ktn]["timestamp_col"] or "(none)"
        print(f"   {ktn} ({ncols} cols, ts: {ts})")

print(f"\nNext: Run 06_Create_Data_Agent.ipynb to create QA and Operations agents.")